# 03 Climate extension M1 and sanity checks (Steps 4, 5, 7)

**M1 = B4s + climate.** Each week's logit gets sum(theta * X), where X are standardised climate **anomalies** (temperature, log(1 + rain), humidity; anomaly = value minus the municipality's harmonic climatology fitted on training years, divided by the residual SD), averaged over 5 lag blocks (0-3, 4-7, 8-11, 12-15, 16-20 weeks) on a continuous weekly timeline. Coefficients are national with shrunken regional deviations.

**Ablations and checks:** M1-short (0-3 weeks only), M1-mult (multinomial likelihood), M1-shufTT (train-and-test climate shuffle, 3 seeds), M1-shufTest (wrong-year climate at test time), zero-climate equivalence, exact sums, uniform-worst, leakage audit, pipeline consistency.

Results are precomputed. `RERUN = True` refits the full weekly pilot (about 22 min on 4 CPUs); `QUICK = True` refits only B4, B4s and M1 (about 5 min) and prints their mean JSD.

In [ ]:
# ---- Setup: works in Google Colab and locally (VS Code / Jupyter) ----
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/aminah9945/dengue-climate-reconstruction"
BRANCH = "main"          # use "official-run" if the pull request is not merged yet
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = Path("/content/dengue-climate-reconstruction")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[all]"], check=True)
    # Optional: keep the downloaded data on Google Drive so it survives the session.
    USE_DRIVE = False
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ.setdefault("DATA_DIR", "/content/drive/MyDrive/dengue_recon_data")
else:
    # Local: this notebook lives in <repo>/notebooks/
    REPO = Path(os.environ.get("DENGUE_ROOT", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())).resolve()

os.environ.setdefault("DENGUE_ROOT", str(REPO))
os.environ.setdefault("DATA_DIR", str(REPO / "data"))
os.environ.setdefault("RESULTS_DIR", str(REPO / "results"))
os.environ.setdefault("XLA_FLAGS", "--xla_force_host_platform_device_count=1")
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))      # makes the package importable without a kernel restart

from dengue_recon import config as C
C.ensure_dirs()
print("repo       :", C.ROOT)
print("DATA_DIR   :", C.DATA_DIR, "| processed data present:", (C.DATA_DIR / "dengue_weekly.parquet").exists())
print("RESULTS_DIR:", C.RESULTS_DIR)

In [ ]:
import json
import numpy as np, pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.width", 200, "display.max_columns", 30)


def show(path, width=900):
    p = Path(path)
    p = p if p.is_absolute() else C.FIG_DIR / p
    display(Image(str(p), width=width)) if p.exists() else print("missing figure:", p)


def have_data():
    ok = (C.DATA_DIR / "dengue_weekly.parquet").exists()
    if not ok:
        print("Processed data not found in DATA_DIR - run notebook 00 first (or set DATA_DIR).")
    return ok

In [ ]:
RERUN, QUICK = False, False
if (RERUN or QUICK) and have_data():
    from dengue_recon.experiments import brazil_loyo
    if QUICK:
        df = brazil_loyo.main("weekly", 60, main_only=True, tag="weekly_60_quick")
    else:
        brazil_loyo.main("weekly", 60)
tm = pd.read_csv(C.BRAZIL_DIR / "table_main_weekly.csv")
display(Markdown(f"**weekly, {tm.note.iloc[0]}**"))
tm[tm.subset == "all"][["model", "n", "jsd_mean", "jsd_median", "peak_err_mean", "emd_mean", "cov80_mean", "cov95_mean", "crps_mean", "jsd_Nweighted"]].round(4)

### Primary contrast (pilot, not a test of H1)
`mean_diff` < 0 means the first model has lower JSD; intervals from municipality and year-block bootstraps.

In [ ]:
p = pd.read_csv(C.BRAZIL_DIR / "table_paired_weekly.csv")
keep = [("M1", "B4s"), ("M1", "B1"), ("M1", "M1-short"), ("M1-shufTT", "B4s"), ("M1-mult", "M1"), ("M2", "B4s")]
sel = p[p.apply(lambda r: (r.model_a, r.model_b) in keep, axis=1) & (p.metric == "jsd")]
sel[["subset", "model_a", "model_b", "n", "mean_diff", "rel_change", "ci_muni_lo", "ci_muni_hi", "ci_year_lo", "ci_year_hi", "n_years_a_better", "headroom_captured"]].round(5)

### Sanity checks

In [ ]:
pd.read_csv(C.BRAZIL_DIR / "sanity_checks.csv")

In [ ]:
show("fig_shuffle.png", 800)

### Where does climate help? (H3: South + Southeast vs North)

In [ ]:
pd.read_csv(C.BRAZIL_DIR / "table_by_region_weekly.csv")[["region", "model_a", "model_b", "n", "mean_diff", "ci_muni_lo", "ci_muni_hi"]].round(5)

In [ ]:
show("fig_gain_by_region_year.png", 1000)

### Lag-block coefficients (left: real climate; right: one shuffled fit)

In [ ]:
show("fig_lag_coefficients.png", 1000)